# 深度学习笔记01：感知器(Perceptron)
> 学习材料：零基础入门深度学习(1)‑感知器
> 核心目标：搞懂最基础神经元——感知器，理解它能干什么、不能干什么，看懂训练规则，读懂Python实现。

##  1. 感知器模型定义
感知器是最早的人工神经元模型，是神经网络的基础单元。
感知器接收多个输入，对输入加权求和后，通过阶跃激活函数输出0或1，用于二分类任务。

### 数学表达式
输入向量：$\boldsymbol{x} = [x_1,x_2,...,x_n]$
权重向量：$\boldsymbol{w} = [w_1,w_2,...,w_n]$
加权和：$z = w_1x_1 + w_2x_2 + ... +w_nx_n + b = \boldsymbol{w}\cdot\boldsymbol{x}+b$

\[
y=f(z)=
\begin{cases}
1 & z>0 \\
0 & z\le0
\end{cases}
\]
- $b$：偏置 bias，作用是平移决策边界，控制激活的阈值
- $f(z)$：阶跃函数，只有0、1两种输出

> 等价写法：把偏置看成一个固定输入为1的权重，$z = w_0\cdot1 + w_1x_1+...+w_nx_n$，$w_0=b$。

### 几何含义
方程 $\boldsymbol{w}\cdot\boldsymbol{x}+b=0$ 在二维平面是一条直线，就是决策边界。
直线一侧输出1，另一侧输出0。
感知器只能用一条直线划分样本，因此只能解决线性可分的二分类。

思考：  
很多教材直接给出公式，容易忽略偏置的意义。如果没有偏置，决策边界一定会经过坐标原点，模型会丢失平移的自由度，很多简单任务都无法拟合。AND门的例子就可以直观看到偏置对直线位置的影响。

## 2. 感知器学习规则
训练目标：寻找一组权重$\boldsymbol{w}$和偏置$b$，让模型预测输出$y$和真实标签$t$尽可能一致。

\begin{aligned}
w_i &\leftarrow w_i + \Delta w_i \\
b &\leftarrow b + \Delta b \\
\Delta w_i &= \eta(t-y)x_i \\
\Delta b &= \eta(t-y)
\end{aligned}

### 参数说明
- $t$：样本真实标签（ground truth）
- $y$：感知器预测输出
- $\eta$：学习率，$0<\eta\le1$，控制每次权重更新的步长

### 更新逻辑
1. 预测正确：$t-y=0$，$\Delta w_i=0$，权重不更新
2. 预测错误：
    - 真实$t=1$，预测$y=0$：$(t-y)=1$，增大权重，让z变大
    - 真实$t=0$，预测$y=1$：$(t-y)=-1$，减小权重，让z变小

### 感知器收敛定理
> 如果训练数据集线性可分，那么感知器学习算法一定会在有限次迭代后收敛，找到可以完全分割样本的决策边界。
> 如果数据集线性不可分，算法永远不会收敛，权重持续震荡。

思考：  
感知器更新规则只在预测出错的时候修改参数。这和后面梯度下降的思路有区别：梯度下降无论预测对错，都会根据损失大小更新参数。同时收敛定理有严格前提。

## 3. 感知器能力边界
### 可解决：线性可分问题
二值逻辑门 AND、OR 都是线性可分，可以用单层感知器实现。
以AND门为例：
| $x_1$ | $x_2$ | 输出y |
|---|---|---|
|0|0|0|
|0|1|0|
|1|0|0|
|1|1|1|
可以找到一条直线，把(1,1)和其余三点分开。

### 不可解决：线性不可分问题
典型例子：XOR异或门
| $x_1$ | $x_2$ | 输出y |
|---|---|---|
|0|0|0|
|0|1|1|
|1|0|1|
|1|1|0|
不存在一条直线，分开两类样本。
单层感知器无法实现异或。

> 历史背景：明斯基在《感知器》书中证明这个局限，让神经网络研究进入第一次AI寒冬。
> 解决方案：使用多层感知器MLP，增加隐藏层，多层叠加可以拟合非线性决策边界。

思考：  
XOR这个例子是理解单层模型局限性的关键。不是感知器本身计算能力弱，而是单层线性划分的表达能力上限就是直线。增加隐藏层之后，多层线性变换叠加，才可以构造出非线性的分割边界。

## 4. Python代码实现
> 实现思路：封装Perceptron类，支持自定义输入维度与激活函数；
> 训练逻辑：逐个样本前向预测，使用感知器规则更新权重与偏置。
> 不使用第三方库，仅使用Python内置reduce函数。

思考：  
将激活函数作为参数传入类中是面向对象设计的优势。后续学习线性单元时，可以直接复用这套训练框架，只替换激活函数，不需要重写训练循环。

In [8]:
from functools import reduce
class Perceptron(object):
    def __init__ (self,input_num,activator):
        self.activator=activator
        self.weights=[0.0 for _ in range (input_num)]
        self.bias=0.0
    def __str__(self):
        return 'weights\t:%s\nbias\t:%f\n' % (self.weights, self.bias)
    def predict(self,input_vec):
        return self.activator(
            reduce(lambda a,b:a+b,
                   map(lambda xw: xw[0] * xw[1],zip(input_vec,self.weights))
                   ,0.0+self.bias)
        )
    def train(self, input_vecs, labels, iteration, rate):
        for i in range(iteration):
            self._one_iteration(input_vecs, labels, rate)

    def _one_iteration(self, input_vecs, labels, rate):
        '''一轮完整遍历全部样本'''
        samples = zip(input_vecs, labels)
        for (input_vec, label) in samples:
            output = self.predict(input_vec)
            self._update_weights(input_vec, output, label, rate)

    def _update_weights(self, input_vec, output, label, rate):
        delta = label - output
        self.weights = list(map(
            lambda xw: xw[1] + rate * delta * xw[0],
            zip(input_vec, self.weights)))
        self.bias += rate * delta

### 实验1：AND与门测试
使用上面Perceptron类，训练AND逻辑。
> 注意：训练得到的权重不唯一，只要决策边界满足分割要求，都是可行解。

思考：  
多次运行代码会发现，每次训练结束得到的权重和偏置数值不一样，但是都可以正确区分四个样本。这说明线性分类的解不是唯一，只要决策边界落在满足分割条件的区域内都有效。

In [9]:
def f(x):
    return 1 if x>0 else 0

def get_training_dataset():
    input_vecs = [[1,1], [0,0], [1,0], [0,1]]
    labels = [1,0,0,0]
    return input_vecs, labels

def train_and_perceptron():
    p = Perceptron(2, f)
    input_vecs, labels = get_training_dataset()
    p.train(input_vecs, labels, 10, 0.1)
    return p

if __name__ == '__main__':
    and_perception = train_and_perceptron()
    print("AND门训练结果：")
    print (and_perception)
    print("1 AND 1 =", and_perception.predict([1,1]))
    print("0 AND 0 =", and_perception.predict([0,0]))
    print("1 AND 0 =", and_perception.predict([1,0]))
    print("0 AND 1 =", and_perception.predict([0,1]))

AND门训练结果：
weights	:[0.1, 0.2]
bias	:-0.200000

1 AND 1 = 1
0 AND 0 = 0
1 AND 0 = 0
0 AND 1 = 0


### 实验2：XOR异或门测试
XOR是线性不可分，观察训练结果。
> 预期现象：无论迭代多少次，模型无法全部预测正确，权重持续震荡。

思考：  
运行代码后会观察到权重一直在来回变化，不会稳定收敛。因为数据集本身无法用直线分割，感知器收敛定理的前提不成立，参数永远在震荡，无法找到一个可以完全分类的决策边界。

In [10]:
def get_xor_data():
    inputs = [[0,0],[0,1],[1,0],[1,1]]
    labels = [0,1,1,0]
    return inputs, labels

p_xor = Perceptron(2, step_activator)
xor_in, xor_label = get_xor_data()
p_xor.train(xor_in, xor_label, iteration=50, rate=0.1)

print("XOR门训练结果：")
print(p_xor)
print("0 XOR 0 =", p_xor.predict([0,0]))
print("0 XOR 1 =", p_xor.predict([0,1]))
print("1 XOR 0 =", p_xor.predict([1,0]))
print("1 XOR 1 =", p_xor.predict([1,1]))

XOR门训练结果：
weights	:[-0.1, 0.0]
bias	:0.100000

0 XOR 0 = 1
0 XOR 1 = 1
1 XOR 0 = 0
1 XOR 1 = 0


正确结果应该分别是0,1,1,0。从结果来看，单层感知器只能解决线性可分问题（AND、OR），而XOR是线性不可分，不存在一条直线把两类样本分开，所以无论怎么调迭代次数、学习率，单层感知器永远无法收敛到完全正确的结果。